# J2 après-midi · 1. Redis depuis Python

Livret de TP J2 après-midi, **exercices 1 et 2**. Exécutez les cellules dans l'ordre
(Maj+Entrée). Les cellules marquées `# À COMPLÉTER` sont à écrire : remplacez les `...`.

Avant de commencer : base 0 vidée et amorce chargée (voir « Avant de commencer » du livret).

## Exercice 1 : Se connecter à Redis

**Question 1.** Exécutez la cellule : elle affiche l'adresse de Redis donnée par le kit, crée la
connexion `r` et envoie un `PING`.

In [2]:
import os
import redis

# Le kit donne l'adresse de Redis dans la variable REDIS_URL
print(os.environ["REDIS_URL"])

# decode_responses=True : Redis renvoie des chaînes Python (str), pas des octets (bytes)
r = redis.from_url(os.environ["REDIS_URL"], decode_responses=True)
r.ping()

redis://redis:6379/0


True

**Question 2.** Chaque commande Redis devient une méthode de `r`, au même nom, en minuscules.
Affichez le nombre de clés de la base (la commande `DBSIZE`).

In [3]:
# À COMPLÉTER : le nombre de clés de la base
r.dbsize()

1013

**Question 3.** Lisez la fiche du restaurant Vella (`HGETALL resto:41704620`). Quel type Python
obtenez-vous ? Affichez ensuite seulement le nom.

In [4]:
# À COMPLÉTER : la fiche complète du restaurant 41704620
fiche = r.hgetall("resto:41704620")
print(type(fiche))
fiche

<class 'dict'>


{'nom': 'Vella',
 'quartier': 'Manhattan',
 'cuisine': 'Italian',
 'adresse': '1480 2 Avenue',
 'dernier_grade': 'A'}

In [5]:
# À COMPLÉTER : le nom, lu dans le dictionnaire fiche
fiche["nom"]

'Vella'

**Question 4.** Sans `decode_responses=True`, que renvoie Redis ? Exécutez la cellule.

In [6]:
r_brut = redis.from_url(os.environ["REDIS_URL"])   # sans decode_responses
r_brut.hget("resto:41704620", "nom")

b'Vella'

**Question 5.** Comptez les sessions sans bloquer le serveur : `r.scan_iter(match=..., count=100)`
parcourt les clés avec `SCAN`, appel après appel, et les donne une par une. N'utilisez pas
`r.keys()`.

In [7]:
# À COMPLÉTER : compter les clés session:* avec r.scan_iter
nb_sessions = sum(1 for _ in r.scan_iter(match="session:*", count=100))
nb_sessions

1000

**Question 6.** Affichez le type de la clé `favoris:lea`, puis ses éléments, triés.
`smembers` renvoie un ensemble Python (`set`), qui n'a pas d'ordre : `sorted` le transforme en
liste triée.

In [9]:
# À COMPLÉTER : le type de favoris:lea, puis ses éléments triés
print(r.type("favoris:lea"))
sorted(r.smembers("favoris:lea"))

set


['40384528', '40402172', '40732665', '41200846', '41704620']

## Exercice 2 : Simuler le trafic de NYC Eats

Les 12 restaurants de l'amorce, avec un poids : plus il est grand, plus le restaurant est
consulté. Exécutez la cellule.

In [10]:
import random
import time

RESTAURANTS = ["41704620", "40732665", "40364335", "40383464", "40402172", "50000923",
               "40384528", "40841272", "41461099", "40397574", "40366132", "41200846"]
POIDS = [3, 10, 8, 6, 5, 7, 2, 4, 3, 2, 1, 9]
PSEUDOS = ["lea", "karim", "sofia", "malik", "ines"]

**Question 1.** Écrivez la fonction `consulter(pseudo, id_resto)` : les commandes d'une
consultation, comme au matin (exercice 10). Elle augmente le compteur `resto:<id>:vues`, ajoute 1
au score du restaurant dans `classement:vues`, et met le restaurant en tête de
`utilisateur:<pseudo>:derniers` en ne gardant que les 5 derniers.

In [11]:
def consulter(pseudo, id_resto):
    """Enregistre une consultation du restaurant id_resto par le visiteur pseudo."""
    # À COMPLÉTER : quatre commandes (incr, zincrby, lpush, ltrim)
    r.incr(f"resto:{id_resto}:vues")
    r.zincrby("classement:vues", 1, id_resto)
    r.lpush(f"utilisateur:{pseudo}:derniers", id_resto)
    r.ltrim(f"utilisateur:{pseudo}:derniers", 0, 4)

**Question 2.** Simulez 1 000 consultations. La graine `2026` fixe le hasard : tout le monde
obtient les mêmes consultations. Combien de temps faut-il ?

In [12]:
alea = random.Random(2026)
debut = time.perf_counter()
for _ in range(1000):
    pseudo = alea.choice(PSEUDOS)
    id_resto = alea.choices(RESTAURANTS, weights=POIDS)[0]
    consulter(pseudo, id_resto)
duree = time.perf_counter() - debut
print(f"1000 consultations en {duree:.2f} s, soit {duree:.2f} ms par consultation")

1000 consultations en 1.84 s, soit 1.84 ms par consultation


**Question 3.** Affichez les 5 restaurants les plus vus : leur rang, leur nom (lu dans la fiche)
et leur nombre de consultations. `zrevrange(..., withscores=True)` renvoie une liste de couples
`(identifiant, score)`, le score étant un nombre à virgule.

In [13]:
podium = r.zrevrange("classement:vues", 0, 4, withscores=True)
for rang, (id_resto, score) in enumerate(podium, start=1):
    # À COMPLÉTER : lire le nom dans la fiche, puis afficher rang, nom et score entier
    nom = r.hget(f"resto:{id_resto}", "nom")
    print(rang, nom, int(score))

1 Katz'S Delicatessen 169
2 Peter Luger Steakhouse 140
3 Lucali 138
4 Di Fara Pizzeria 123
5 Le Bernardin 109


**Question 4.** Affichez les noms des derniers restaurants consultés par Léa, du plus récent au
plus ancien.

In [14]:
# À COMPLÉTER : la liste utilisateur:lea:derniers, puis le nom de chaque restaurant
derniers_lea = r.lrange("utilisateur:lea:derniers", 0, -1)
[r.hget(f"resto:{id_resto}", "nom") for id_resto in derniers_lea]

["Lombardi'S",
 'Lucali',
 "Katz'S Delicatessen",
 'Peter Luger Steakhouse',
 "Katz'S Delicatessen"]

**Question 5.** Les compteurs et le classement racontent-ils la même histoire ? Faites la somme
des 12 compteurs, puis celle des scores du classement.

In [15]:
# À COMPLÉTER : somme des compteurs (r.get renvoie None si la clé n'existe pas : "or 0")
total_compteurs = sum(int(r.get(f"resto:{id_resto}:vues") or 0) for id_resto in RESTAURANTS)
total_classement = sum(score for _, score in r.zrange("classement:vues", 0, -1, withscores=True))
print(total_compteurs, total_classement)

1000 1000.0
